# RoadVision AI — Inference and Predictions

Runs a trained detector on new road images or videos and visualizes the detected vehicles.

**Configuration (env vars):**
- `RV_SOURCE`: image file, folder of images, or video file. If unset, six validation images are used as a demo (they are not new data).
- `RV_WEIGHTS_PATH`: weights. Default: best row of `reports/experiment_comparison.csv`, else the baseline `best.pt`.
- `RV_CONF` (0.25), `RV_IOU` (0.7), `RV_IMGSZ` (training size), `RV_TRACK=1` (ByteTrack IDs and unique counts for video), `RV_VID_STRIDE` (1), `RV_MAX_FRAMES` (0 = all).

**Constraints:** rare classes (policecar, scooter, minibus, wheelbarrow) are weak in validation; low confidence on them is expected. Outputs go to `predictions/<timestamp>/`. Source files are never modified.

In [ ]:
# %pip install -q ultralytics

In [ ]:
from pathlib import Path
import os, json, random, time, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2, torch
from ultralytics import YOLO
from IPython.display import display

SEED = 42
random.seed(SEED); np.random.seed(SEED)

DEFAULT_ROOT = Path.home() / ".cache/kagglehub/datasets/ashfakyeafi/road-vehicle-images-dataset/versions/2/roadvision_cleaned"
CLEAN_ROOT = Path(os.getenv("ROADVISION_CLEANED_ROOT", str(DEFAULT_ROOT))).expanduser().resolve()
if not CLEAN_ROOT.is_dir():
    raise FileNotFoundError(f"Prepared dataset not found: {CLEAN_ROOT}. Set ROADVISION_CLEANED_ROOT.")
DATA_YAML = CLEAN_ROOT / "roadvision_prepared.yaml"  # produced by 04_dataset_preparation.ipynb
if not DATA_YAML.is_file():
    raise FileNotFoundError(f"{DATA_YAML} not found. Run 04_dataset_preparation.ipynb first.")
_cfg = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
_names = _cfg["names"]
CLASS_NAMES = [_names[k] for k in sorted(_names, key=int)] if isinstance(_names, dict) else list(_names)
NC = len(CLASS_NAMES)
SPLITS = {s: CLEAN_ROOT / s for s in ("train", "valid", "test")}
RUNS_DIR = CLEAN_ROOT / "training_runs"
REPORT_DIR = CLEAN_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
WORKERS = 2
MIN_EVAL_INSTANCES = 20  # classes with fewer val instances give statistically meaningless AP

def count_instances(label_dir, nc):
    counts = np.zeros(nc, dtype=int)
    for f in Path(label_dir).glob("*.txt"):
        for line in f.read_text(encoding="utf-8").splitlines():
            parts = line.split()
            if len(parts) == 5: counts[int(float(parts[0]))] += 1
    return counts

def resolve_weights():
    env = os.getenv("RV_WEIGHTS_PATH")
    if env: return Path(env).expanduser()
    csv = REPORT_DIR / "experiment_comparison.csv"
    if csv.is_file():
        df = pd.read_csv(csv)
        df = df[df["status"] == "ok"].sort_values("mAP50-95", ascending=False)
        for w in df["best_weights"]:
            if Path(w).is_file(): return Path(w)
    return RUNS_DIR / "yolov8n_baseline" / "weights" / "best.pt"

WEIGHTS = resolve_weights()
if not WEIGHTS.is_file(): raise FileNotFoundError(f"Weights not found: {WEIGHTS}")
model = YOLO(str(WEIGHTS))
try: IMGSZ = int(os.getenv("RV_IMGSZ", model.ckpt["train_args"]["imgsz"]))
except Exception: IMGSZ = 640
CONF = float(os.getenv("RV_CONF", 0.25))
IOU = float(os.getenv("RV_IOU", 0.7))
TRACK = os.getenv("RV_TRACK", "0") == "1"
VID_STRIDE = max(1, int(os.getenv("RV_VID_STRIDE", 1)))
MAX_FRAMES = int(os.getenv("RV_MAX_FRAMES", 0))
OUT_DIR = CLEAN_ROOT / "predictions" / time.strftime("%Y%m%d_%H%M%S")
(OUT_DIR / "images").mkdir(parents=True, exist_ok=True)
print("Weights:", WEIGHTS, "| imgsz:", IMGSZ, "| conf:", CONF, "| iou:", IOU)
print("Output:", OUT_DIR)

## 1. Resolve the source

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
VID_EXT = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".m4v"}

SOURCE = os.getenv("RV_SOURCE")
if SOURCE:
    src = Path(SOURCE).expanduser()
    if not src.exists(): raise FileNotFoundError(f"Source not found: {src}")
    if src.is_dir():
        image_paths = sorted(p for p in src.rglob("*") if p.suffix.lower() in IMG_EXT)
        video_paths = sorted(p for p in src.rglob("*") if p.suffix.lower() in VID_EXT)
    else:
        image_paths = [src] if src.suffix.lower() in IMG_EXT else []
        video_paths = [src] if src.suffix.lower() in VID_EXT else []
else:
    print("RV_SOURCE not set: demo on 6 validation images (not new data).")
    val_imgs = sorted(p for p in (SPLITS["valid"] / "images").iterdir() if p.suffix.lower() in IMG_EXT)
    image_paths, video_paths = random.Random(SEED).sample(val_imgs, min(6, len(val_imgs))), []
print(f"Images: {len(image_paths)} | Videos: {len(video_paths)}")
if not image_paths and not video_paths: raise ValueError("No supported image or video files found.")

## 2. Image predictions
Saves annotated images and one row per detection.

In [ ]:
det_rows = []
shown = []
for p in image_paths:
    r = model.predict(str(p), imgsz=IMGSZ, conf=CONF, iou=IOU, device=DEVICE, verbose=False)[0]
    for c, xy, s in zip(r.boxes.cls.tolist(), r.boxes.xyxy.tolist(), r.boxes.conf.tolist()):
        det_rows.append({"file": p.name, "class_id": int(c), "class": CLASS_NAMES[int(c)], "conf": round(float(s), 4),
                         "x1": round(xy[0], 1), "y1": round(xy[1], 1), "x2": round(xy[2], 1), "y2": round(xy[3], 1)})
    annotated = r.plot()  # BGR with boxes and labels
    cv2.imwrite(str(OUT_DIR / "images" / f"{p.stem}_pred.jpg"), annotated)
    if len(shown) < 12: shown.append((p.name, cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB), len(r.boxes)))

det_df = pd.DataFrame(det_rows, columns=["file", "class_id", "class", "conf", "x1", "y1", "x2", "y2"])
det_df.to_csv(OUT_DIR / "detections_images.csv", index=False)
print(f"{len(det_df)} detections in {len(image_paths)} images -> {OUT_DIR / 'detections_images.csv'}")

if shown:
    cols = min(3, len(shown)); rows = int(np.ceil(len(shown) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 5 * rows), squeeze=False)
    for ax in axes.ravel(): ax.axis("off")
    for ax, (name, img, n) in zip(axes.ravel(), shown):
        ax.imshow(img); ax.set_title(f"{name} — {n} vehicles", fontsize=9)
    plt.tight_layout(); plt.show()

## 3. Image summary

In [ ]:
if det_df.empty:
    print("No detections. Lower RV_CONF or check that the images show road vehicles.")
else:
    counts = det_df["class"].value_counts()
    ax = counts.sort_values().plot.barh(figsize=(8, 0.4 * len(counts) + 2)); ax.set_title(f"Detections by class (conf >= {CONF})")
    plt.tight_layout(); plt.show()
    per_img = det_df.groupby("file").size().rename("vehicles").to_frame()
    display(per_img.join(det_df.groupby("file")["conf"].mean().rename("mean_conf").round(3)))
    low = det_df[det_df.conf < 0.4]
    print(f"Low-confidence detections (<0.4): {len(low)} of {len(det_df)}")

## 4. Video predictions
Writes an annotated `.mp4` (codec `mp4v`; open it in a desktop player, not necessarily in the browser) and per-frame counts. With `RV_TRACK=1`, ByteTrack assigns IDs so unique vehicles per class are counted, not per-frame detections.

In [ ]:
def process_video(path):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened(): print("Cannot open:", path); return None
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out_path = OUT_DIR / f"{path.stem}_pred.mp4"
    writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), fps / VID_STRIDE, (w, h))
    frame_rows, samples, seen_ids, i, done = [], [], {}, -1, 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        i += 1
        if i % VID_STRIDE: continue
        if TRACK:
            r = model.track(frame, persist=True, tracker="bytetrack.yaml", imgsz=IMGSZ, conf=CONF, iou=IOU, device=DEVICE, verbose=False)[0]
        else:
            r = model.predict(frame, imgsz=IMGSZ, conf=CONF, iou=IOU, device=DEVICE, verbose=False)[0]
        cls = [int(c) for c in r.boxes.cls.tolist()]
        row = {"frame": i, "vehicles": len(cls)}
        for c in set(cls): row[CLASS_NAMES[c]] = cls.count(c)
        frame_rows.append(row)
        if TRACK and r.boxes.id is not None:
            for tid, c in zip(r.boxes.id.int().tolist(), cls): seen_ids.setdefault(tid, c)
        annotated = r.plot()
        writer.write(annotated)
        if done % max(1, 60 // VID_STRIDE) == 0 and len(samples) < 6: samples.append((i, cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)))
        done += 1
        if MAX_FRAMES and done >= MAX_FRAMES: break
    cap.release(); writer.release()
    df = pd.DataFrame(frame_rows).fillna(0)
    df.to_csv(OUT_DIR / f"{path.stem}_frame_counts.csv", index=False)
    print(f"{path.name}: {done} frames processed -> {out_path}")
    if TRACK:
        uniq = pd.Series([CLASS_NAMES[c] for c in seen_ids.values()]).value_counts()
        print("Unique tracked vehicles by class:"); display(uniq.to_frame("unique_vehicles"))
        uniq.to_csv(OUT_DIR / f"{path.stem}_unique_tracks.csv")
    return df, samples

if not video_paths:
    print("No video in the source. Skipped.")
for vp in video_paths:
    result = process_video(vp)
    if result is None: continue
    df, samples = result
    if samples:
        fig, axes = plt.subplots(2, 3, figsize=(18, 8), squeeze=False)
        for ax in axes.ravel(): ax.axis("off")
        for ax, (fi, img) in zip(axes.ravel(), samples): ax.imshow(img); ax.set_title(f"frame {fi}", fontsize=9)
        plt.tight_layout(); plt.show()
    if not df.empty:
        df.plot(x="frame", y="vehicles", figsize=(10, 3), title=f"Vehicles per frame — {vp.name}"); plt.tight_layout(); plt.show()

## 5. Notes
- Per-frame counts without tracking count the same vehicle in every frame; use `RV_TRACK=1` for unique counts.
- The detector is trained on one dataset's road scenes (vehicle types such as rickshaw and CNG three-wheelers). Expect weaker results under different cameras, weather, night or other regions; verify on a handful of your own images before relying on it.
- Run `08_model_evaluation.ipynb` first to know which classes can be trusted.